# T1-08b PDF, vector store, then generate

**Time:** after T1-08 (extra) &nbsp;|&nbsp; **Evidence:** a cited reply from `CD-COURIER-02.pdf` in `logs/`

**Notebook learning outcomes**

- Extract text from a **born-digital PDF** (this is not OCR).
- Chunk it, **embed** it, and store the chunks in a local **vector database**.
- Send the retrieved chunks through the same Day 1 generate path as T1-08.

Run **T1-08 first**. That notebook used a Markdown file and a keyword index. This one keeps the same pipeline and changes the engine: PDF extract + cosine search.

**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Overnight readers — same job, different engine

T1-08 already showed: source → chunks → index → prompt → generate (or labelled fallback).

| Step | T1-08 | This notebook |
|---|---|---|
| Source | UTF-8 Markdown | Born-digital **PDF** |
| Extract | Read the text file | `pypdf` pulls the text out of the page |
| Index | In-memory keyword postings | **Chroma** stores vectors on disk |
| Score | Shared words | Cosine similarity |
| Generate | `civicdesk_rag.generate_with_rag` | The same function |

A PDF is not automatically a chunk. A vector database is not permission. A cosine score is not a citation.

This still uses **CivicDesk fiction only**. Do not drop in a ministry PDF.

**On the course Docker image:** `pypdf`, `chromadb`, and `sentence-transformers` are already installed. MiniLM (`all-MiniLM-L6-v2`) is baked at image build into `/models/hf-cache`. Embed and store stay on this machine; the Hub is off at runtime (`HF_HUB_OFFLINE=1`). **Laptop or Colab without that rebuilt image:** use the optional install cell; MiniLM then downloads once (~91 MB) into the local Hugging Face cache.


## Ministry link

`CD-COURIER-02` stands in for a short circular on **paper originals** leaving the building (Finance registry / RDLG desk / any office that still moves signed letters).

| Question | What you should see |
|---|---|
| Original in a private car overnight | Cite the pouch rule |
| Pouch shelf empty | Cite **stop**; do not invent a workaround |
| Email a scan home as a backup | Cite that this circular does not authorise that |

Labour / Health still do not hire or publish from the model. This notebook does not reopen the handbook JSON.


## 0. Backend — demo, OLMo, or Phi

Retrieval always runs on this machine. This cell only chooses **what generates**.

| `BACKEND` | Use when |
|---|---|
| **`demo`** | No GPU, overnight, or the server is down. Canned / template replies. **Not** a live-model score. |
| **`olmo`** | Local OLMo is up (T1-01a / `docker compose`, port **8000**). Needs VRAM. |
| **`phi`** | Local Phi is up (port **8001**). Compose loads OLMo only unless you set `CIVICDESK_LOAD_MODELS=olmo,phi`. |

Change the one marked line and re-run **this** cell. Later generate cells in this kernel follow it. If the live call fails, the notebook continues with a labelled local fallback.


In [ ]:
# [CHANGE] where generate() goes. Retrieval still runs here either way.
BACKEND = "demo"  # "demo" | "olmo" | "phi"

import civicdesk_rag as rag

rag.set_backend(BACKEND)


## 0. Packages


In [ ]:
# [RUN] what this extra needs
import importlib.util

needed = [
    ("pypdf", "pypdf"),
    ("chromadb", "chromadb"),
    ("sentence_transformers", "sentence-transformers"),
]
missing = []
for mod, pip_name in needed:
    ok = importlib.util.find_spec(mod) is not None
    print(f"{pip_name:24} {'ok' if ok else 'MISSING'}")
    if not ok:
        missing.append(pip_name)
if missing:
    print("Run the next optional cell, or in a terminal:")
    print("  pip install " + " ".join(missing))
else:
    print("Ready to extract, embed and store.")
    print("On the course image MiniLM is already in HF_HOME; this cell did not download weights.")


In [ ]:
# [OPTIONAL] install only what is missing. Skip on the course image (Ready above).
import sys
import subprocess
import importlib.util

pkgs = []
if importlib.util.find_spec("pypdf") is None:
    pkgs.append("pypdf")
if importlib.util.find_spec("chromadb") is None:
    pkgs.append("chromadb")
if importlib.util.find_spec("sentence_transformers") is None:
    pkgs.append("sentence-transformers")
if not pkgs:
    print("Nothing to install.")
else:
    print("installing", pkgs)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs])


## 1. Check the model you are about to use


In [ ]:
# [RUN]
import civicdesk_rag as rag

cfg = rag.endpoint_probe()
print("backend:", cfg["backend"], "mode:", cfg["mode"], "model:", cfg["model"])
print("timeouts:", cfg["timeouts"])
print(cfg["note"])
if cfg.get("offline"):
    print(cfg["offline"])
print(cfg.get("local_vs_host", ""))


## 2. Point to the PDF

This file is a **source document**, not `handbooks.json`. Extraction is PDF text extract. It is not the T1-07 scan/OCR path.


In [ ]:
# [RUN]
from pathlib import Path
import json
import civicdesk_rag as rag

SOURCE_PATH = Path("data/source_documents/CD-COURIER-02.pdf")
print("SOURCE_PATH:", SOURCE_PATH.resolve())
print("exists:", SOURCE_PATH.is_file(), "suffix:", SOURCE_PATH.suffix)
print("--- intake ---")
print((SOURCE_PATH.with_name(SOURCE_PATH.stem + ".intake.json")).read_text(encoding="utf-8"))
print("This is a born-digital PDF. T1-07's PNG still needs checked OCR before it may be indexed.")


### Read and extract

`pypdf` reads strings already in the file. It does not look at pixels. A scanned page with no text layer would come out empty — that is when T1-07's OCR checklist applies.


In [ ]:
# [RUN]
import json
import civicdesk_rag as rag

doc = rag.load_pdf_document(SOURCE_PATH)
print("title:", doc["title"], "version:", doc["version"])
print("extraction:", doc["extraction"], "pages:", doc["n_pages"])
for p in doc["pages"]:
    print("=" * 40, "page", p["page"])
    print(p["text"])
print("--- full extracted text length ---", len(doc["text"]))


## 3. Chunk (keep provenance)


In [ ]:
# [CHANGE] chunk size. Rebuild after you change it.
MAX_WORDS = 45
OVERLAP = 10

import civicdesk_rag as rag

PDF_CHUNKS = rag.chunk_source(doc, max_words=MAX_WORDS, overlap=OVERLAP)
for c in PDF_CHUNKS:
    print(f"{c['id']} page={c.get('page')} seq={c['sequence']} words={len(c['text'].split())}")
    print("  path:", c["source_path"], "ver:", c["version"], "clearance:", c["clearance"])
    print(" ", c["text"][:180])
    print()
print(len(PDF_CHUNKS), "chunks from", SOURCE_PATH.name)
assert all("CD-COURIER-02" in (c.get("source_path") or "") or c["id"].startswith("CD-COURIER-02") for c in PDF_CHUNKS)
print("Every chunk still carries the PDF id / path. These are not handbook chunks.")


## 4. Store in a vector database

Chroma here is a **local** store under `data/vector_store/`. It is not a ministry EDMS. Each record is: id, text, metadata, and a MiniLM vector. MiniLM is an **embedding** model (retrieve); OLMo/Phi still **generate**. On the course image those MiniLM weights are already on disk — this cell must not reach Hugging Face. The generator still has not seen this text.


In [ ]:
# [RUN]
from pathlib import Path
import os
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
import civicdesk_rag as rag

print("HF_HOME:", os.environ.get("HF_HOME", "(default cache)"))
print("HF_HUB_OFFLINE:", os.environ.get("HF_HUB_OFFLINE", "(unset)"))

STORE = Path("data/vector_store/t1_08b")
STORE.mkdir(parents=True, exist_ok=True)
COLLECTION = "cd_courier_02"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embed_fn = SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
client = chromadb.PersistentClient(path=str(STORE))
try:
    client.delete_collection(COLLECTION)
except Exception:
    pass
coll = client.get_or_create_collection(name=COLLECTION, embedding_function=embed_fn)

coll.add(
    ids=[c["id"] for c in PDF_CHUNKS],
    documents=[c["text"] for c in PDF_CHUNKS],
    metadatas=[
        {
            "source_path": str(c.get("source_path") or ""),
            "version": str(c.get("version") or ""),
            "page": int(c.get("page") or 0),
            "sequence": int(c["sequence"]),
            "clearance": str(c.get("clearance") or ""),
            "citation": str(c.get("citation") or ""),
            "title": str(c.get("title") or ""),
            "index_ok": str(bool(c.get("index_ok", True))),
        }
        for c in PDF_CHUNKS
    ],
)
print("stored", coll.count(), "chunks in", STORE)
print("embedding model:", EMBED_MODEL)
print("A stored vector is not yet a prompt, and not yet a permission.")
print("If HF_HUB_OFFLINE=1 this used the baked/cached MiniLM files only.")
BY_ID = {c["id"]: c for c in PDF_CHUNKS}


## 5. Query the vector store


In [ ]:
# [CHANGE] the clerk question. Then try the paraphrase.
QUESTION = "May I take a leave letter original home in my car overnight if the pouch shelf is empty?"
PARAPHRASE = "Can I drop staff appointment papers at my house if courier bags have run out?"
ROLE = "clerk"
K = 3

import json

def query_store(question, k=K):
    """Always read the named collection from disk. Do not reuse a deleted handle."""
    try:
        store = client.get_collection(COLLECTION, embedding_function=embed_fn)
    except Exception:
        print("(collection missing)")
        return []
    n = store.count()
    if n == 0:
        print("(collection empty — no hits)")
        return []
    raw = store.query(
        query_texts=[question],
        n_results=min(k, n),
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for i, cid in enumerate(raw["ids"][0]):
        if cid not in BY_ID:
            continue
        row = dict(BY_ID[cid])
        row["distance"] = raw["distances"][0][i]
        row["score"] = 1.0 - float(row["distance"])
        hits.append(row)
    return hits

print("USER:", QUESTION)
hits = query_store(QUESTION)
print(json.dumps([{"id": h["id"], "page": h.get("page"), "distance": round(h["distance"], 4), "score": round(h["score"], 4)} for h in hits], indent=2))
visible, blocked = rag._gate_hits(hits, ROLE)
print("visible", [v["id"] for v in visible], "blocked", [(b["id"], b["reason"]) for b in blocked])

print()
print("PARAPHRASE:", PARAPHRASE)
ph = query_store(PARAPHRASE)
print("paraphrase hits:", [{"id": h["id"], "distance": round(h["distance"], 4)} for h in ph])


### Inspect the prompt

Same `build_messages` as T1-08. The PDF supplied context. It did not add weights.


In [ ]:
# [RUN]
import civicdesk_rag as rag

hits = query_store(QUESTION)
visible, _ = rag._gate_hits(hits, ROLE)
PDF_MESSAGES = rag.build_messages(QUESTION, visible)
print("chunks in this prompt:", [h["id"] for h in visible] or "(none)")
for m in PDF_MESSAGES:
    print("=" * 40, m["role"])
    print(m["content"])
print()
print("prompt token count (Day 1 tokenizer if present):", rag.prompt_token_count(PDF_MESSAGES))
print("messages_status will be 'sent' only if the backend call succeeds.")


## 6. Generate with the retrieved PDF chunks


In [ ]:
# [RUN]
import civicdesk_rag as rag

hits = query_store(QUESTION)
visible, blocked = rag._gate_hits(hits, ROLE)
print("hits from the store now:", [h["id"] for h in hits] or "(none)")
print("blocked:", [(b["id"], b["reason"]) for b in blocked])

out = rag.generate_with_rag(QUESTION, role=ROLE, chunks_override=visible)
print("mode:", out["mode"], "chat_mode:", out.get("chat_mode"), "evidence:", out["evidence"])
print("generation_succeeded:", out["generation_succeeded"])
print("messages_status:", out["messages_status"])
print(out["reply"])
print("checks:", out["checks"])
print("trace:", SOURCE_PATH.name, "->", [c["id"] for c in PDF_CHUNKS], "->", out["visible_ids"], "->", out["mode"])
rag.log_record(out, name="T1-08b_courier")
if out["mode"] != "live":
    print("not a model answer, not scored")


## 7. Prove the connection

Query, inspect, and generate all read **the vector store**. There is no leftover `PDF_VISIBLE` list.

1. Run the **delete** cell. Count must be 0.
2. Re-run **query** and **generate** above. Hits and `CD-COURIER` in the prompt must disappear.
3. Only then run **restore**.

An answer may still be guessed from model weights — judge whether the citation is gone, not whether the wording changed.


In [ ]:
# [RUN] delete. Then re-run query and generate above. Do not run restore yet.
import civicdesk_rag as rag

client.delete_collection(COLLECTION)
coll = client.get_or_create_collection(name=COLLECTION, embedding_function=embed_fn)
print("count after delete:", coll.count())
print("hits after delete:", [h["id"] for h in query_store(QUESTION)] or "(none)")
msgs = rag.build_messages(QUESTION, [])
print("CD-COURIER in messages:", "CD-COURIER" in msgs[1]["content"])
print("Store is empty. Re-run the query cell and the generate cell above.")
print("Judge provenance, not a promised wording change.")


In [ ]:
# [RUN] only after you have re-run query and generate on the empty store
coll.add(
    ids=[c["id"] for c in PDF_CHUNKS],
    documents=[c["text"] for c in PDF_CHUNKS],
    metadatas=[{"source_path": str(c.get("source_path") or ""), "page": int(c.get("page") or 0)} for c in PDF_CHUNKS],
)
print("restored count:", coll.count())
print("hits:", [h["id"] for h in query_store(QUESTION)])
print("Chunks are back. Re-run generate if you want the cited reply again.")


### After you run

Do not expect a particular sentence from the model. Name any **unknown citation**, **altered quote**, or **wrong NOT IN SOURCES**.

**Write-up:** copy one quoted span that you can point at in the PDF extract. If you cannot point at a span, you do not have RAG — you have a vector store.


In [ ]:
# [OPTIONAL] keyword overlap on the same PDF chunks vs cosine
import civicdesk_rag as rag

kw = rag.retrieve(QUESTION, k=3, pool=PDF_CHUNKS, min_score=1)
print("keyword hits:", [{"id": h["id"], "score": h["score"]} for h in kw])
print("chroma hits:", [{"id": h["id"], "distance": round(h["distance"], 4)} for h in hits])
print("Same chunks, two engines. Neither score is permission.")
